# Evaluación del Expert 3 — DenseNet3D (LUNA16)

Evaluación completa del modelo DenseNet3D entrenado para clasificación binaria de nódulos pulmonares.

- **Checkpoint:** época 23 (entrenamiento detenido por error NCCL en época 23/100)
- **Métricas reportadas en entrenamiento:** F1-Macro=0.9438, AUC=0.9911 (validación)
- **Objetivo:** verificar métricas en val, evaluar en test, detectar posible data drift

In [12]:
# ── Imports ──────────────────────────────────────────────────────────
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

# ── Paths ────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'pipeline'))
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'pipeline' / 'fase2'))
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'pipeline' / 'fase2' / 'models'))

CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'expert_03_densenet3d' / 'best.pt'
LOG_PATH = PROJECT_ROOT / 'checkpoints' / 'expert_03_densenet3d' / 'expert3_ddp_training_log.json'
PATCHES_BASE = PROJECT_ROOT / 'datasets' / 'luna_lung_cancer' / 'patches'
OUTPUT_DIR = PROJECT_ROOT / 'notebooks/evaluacion_expert3'
OUTPUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Output: {OUTPUT_DIR}')

Dispositivo: cpu
Checkpoint: /home/nicolas/Escritorio/Academico/analitica_datos/carlos_andres_ferro/proyecto_2/checkpoints/expert_03_densenet3d/best.pt
Output: /home/nicolas/Escritorio/Academico/analitica_datos/carlos_andres_ferro/proyecto_2/notebooks/evaluacion_expert3


## 1. Carga del modelo desde checkpoint

In [13]:
from expert3_densenet3d import Expert3DenseNet3D

# Cargar checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
print(f"Época del checkpoint: {ckpt['epoch']}")
print(f"Val Loss: {ckpt['val_loss']:.6f}")
print(f"Val F1-Macro: {ckpt['val_f1_macro']:.4f}")
print(f"Val AUC: {ckpt['val_auc']:.4f}")
print(f"Config: {ckpt['config']}")

# Instanciar modelo con la misma config del entrenamiento
cfg = ckpt['config']
model = Expert3DenseNet3D(
    in_channels=1,
    num_classes=2,
    growth_rate=32,
    block_layers=[4, 8, 16, 12],
    init_features=64,
    spatial_dropout_p=cfg['spatial_dropout_3d'],
    fc_dropout_p=cfg['dropout_fc'],
)

# Cargar pesos (el checkpoint puede tener prefijo 'module.' de DDP)
state_dict = ckpt['model_state_dict']
# Remover prefijo 'module.' si existe (guardado desde DDP)
cleaned_state = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(cleaned_state)

model = model.to(DEVICE)
model.eval()
print(f"\nModelo cargado: {model.count_parameters():,} parámetros")
print("Modelo en modo eval()")

Época del checkpoint: 23
Val Loss: 0.006632
Val F1-Macro: 0.9438
Val AUC: 0.9911
Config: {'lr': 0.0003, 'weight_decay': 0.03, 'focal_gamma': 2.0, 'focal_alpha': 0.85, 'label_smoothing': 0.05, 'dropout_fc': 0.4, 'spatial_dropout_3d': 0.15, 'batch_size': 8, 'accumulation_steps': 4, 'fp16': True, 'seed': 42, 'world_size': 2}

Modelo cargado: 6,729,474 parámetros
Modelo en modo eval()


## 2. Carga de datos (val y test)

In [14]:
# ── DataLoader ligero sin dependencias externas (cv2, etc.) ────────
# Se evita importar dataloader_expert3 porque su cadena de imports
# pasa por datasets/__init__.py → chest.py → cv2 (no instalado).
from torch.utils.data import Dataset

# Cargar mapa de labels desde candidates CSV
_CSV_V2 = PATCHES_BASE.parent / 'candidates_V2' / 'candidates_V2.csv'
_CSV_V1 = PATCHES_BASE.parent / 'candidates.csv'
_csv_path = _CSV_V2 if _CSV_V2.exists() else _CSV_V1
_cand_df = pd.read_csv(_csv_path)
_label_map = dict(zip(_cand_df.index, _cand_df['class']))
print(f'CSV: {_csv_path.name} ({len(_cand_df):,} candidatos)')

class _LUNA16Simple(Dataset):
    """Dataset mínimo: lee .npy y devuelve (tensor[1,64,64,64], label, stem)."""
    def __init__(self, patches_dir, label_map):
        self.samples = []
        for f in sorted(Path(patches_dir).glob('candidate_*.npy')):
            try:
                idx = int(f.stem.split('_')[1])
            except (IndexError, ValueError):
                continue
            lbl = label_map.get(idx, -1)
            if lbl >= 0:
                self.samples.append((f, lbl))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        vol = np.load(path).astype(np.float32)
        return torch.from_numpy(vol).unsqueeze(0), label, path.stem

val_loader = DataLoader(
    _LUNA16Simple(PATCHES_BASE / 'val', _label_map),
    batch_size=8, shuffle=False, num_workers=4, pin_memory=True,
)
test_loader = DataLoader(
    _LUNA16Simple(PATCHES_BASE / 'test', _label_map),
    batch_size=8, shuffle=False, num_workers=4, pin_memory=True,
)

print(f"\nVal:  {len(val_loader.dataset):,} muestras, {len(val_loader)} batches")
print(f"Test: {len(test_loader.dataset):,} muestras, {len(test_loader)} batches")

CSV: candidates_V2.csv (754,975 candidatos)

Val:  1,155 muestras, 145 batches
Test: 2,013 muestras, 252 batches


## 3. Inferencia

In [15]:
def run_inference(model, loader, device):
    """Ejecuta inferencia y retorna labels, predicciones y probabilidades."""
    all_labels = []
    all_probs = []
    
    model.eval()
    with torch.no_grad():
        for batch_idx, (volumes, labels, _stems) in enumerate(loader):
            volumes = volumes.to(device)
            logits = model(volumes)  # [B, 2]
            probs = F.softmax(logits, dim=1)[:, 1]  # prob de clase positiva
            
            all_labels.append(labels.numpy())
            all_probs.append(probs.cpu().numpy())
            
            if (batch_idx + 1) % 50 == 0:
                print(f'  Batch {batch_idx + 1}/{len(loader)}')
    
    labels = np.concatenate(all_labels)
    probs = np.concatenate(all_probs)
    preds = (probs >= 0.5).astype(int)
    return labels, preds, probs

print('Inferencia en validación...')
val_labels, val_preds, val_probs = run_inference(model, val_loader, DEVICE)
print(f'  Completado: {len(val_labels)} muestras')

print('\nInferencia en test...')
test_labels, test_preds, test_probs = run_inference(model, test_loader, DEVICE)
print(f'  Completado: {len(test_labels)} muestras')

Inferencia en validación...
  Batch 50/145
  Batch 100/145
  Completado: 1155 muestras

Inferencia en test...
  Batch 50/252
  Batch 100/252
  Batch 150/252
  Batch 200/252
  Batch 250/252
  Completado: 2013 muestras


## 4. Cálculo de métricas

In [16]:
def compute_metrics(labels, preds, probs):
    """Calcula todas las métricas de evaluación."""
    return {
        'F1-Macro': f1_score(labels, preds, average='macro'),
        'F1-Binary': f1_score(labels, preds, average='binary'),
        'Accuracy': accuracy_score(labels, preds),
        'Precision': precision_score(labels, preds),
        'Recall': recall_score(labels, preds),
        'AUC-ROC': roc_auc_score(labels, probs),
        'N_samples': len(labels),
        'N_positives': int(labels.sum()),
        'N_negatives': int((labels == 0).sum()),
    }

val_metrics = compute_metrics(val_labels, val_preds, val_probs)
test_metrics = compute_metrics(test_labels, test_preds, test_probs)

# Tabla resumen
metrics_df = pd.DataFrame({
    'Métrica': list(val_metrics.keys()),
    'Validación': [f'{v:.4f}' if isinstance(v, float) else str(v) for v in val_metrics.values()],
    'Test': [f'{v:.4f}' if isinstance(v, float) else str(v) for v in test_metrics.values()],
})

# Agregar columna de diferencia para métricas numéricas float
diffs = []
for k in val_metrics:
    v, t = val_metrics[k], test_metrics[k]
    if isinstance(v, float):
        diffs.append(f'{t - v:+.4f}')
    else:
        diffs.append('—')
metrics_df['Δ (Test-Val)'] = diffs

print('\n' + '=' * 65)
print('  TABLA RESUMEN DE MÉTRICAS — Expert 3 DenseNet3D')
print('=' * 65)
print(metrics_df.to_string(index=False))
print('=' * 65)

# Comparar con métricas reportadas en entrenamiento
print(f"\n--- Verificación vs checkpoint ---")
print(f"F1-Macro checkpoint: {ckpt['val_f1_macro']:.4f} | Recalculado: {val_metrics['F1-Macro']:.4f}")
print(f"AUC checkpoint:      {ckpt['val_auc']:.4f} | Recalculado: {val_metrics['AUC-ROC']:.4f}")


  TABLA RESUMEN DE MÉTRICAS — Expert 3 DenseNet3D
    Métrica Validación   Test Δ (Test-Val)
   F1-Macro     0.9438 0.9353      -0.0085
  F1-Binary     0.8981 0.8820      -0.0161
   Accuracy     0.9810 0.9791      -0.0018
  Precision     0.8739 0.9075      +0.0336
     Recall     0.9238 0.8579      -0.0659
    AUC-ROC     0.9911 0.9821      -0.0090
  N_samples       1155   2013            —
N_positives        105    183            —
N_negatives       1050   1830            —

--- Verificación vs checkpoint ---
F1-Macro checkpoint: 0.9438 | Recalculado: 0.9438
AUC checkpoint:      0.9911 | Recalculado: 0.9911


In [17]:
# Classification reports detallados
print('\n=== Classification Report — Validación ===')
print(classification_report(val_labels, val_preds, target_names=['No nódulo', 'Nódulo']))

print('\n=== Classification Report — Test ===')
print(classification_report(test_labels, test_preds, target_names=['No nódulo', 'Nódulo']))


=== Classification Report — Validación ===
              precision    recall  f1-score   support

   No nódulo       0.99      0.99      0.99      1050
      Nódulo       0.87      0.92      0.90       105

    accuracy                           0.98      1155
   macro avg       0.93      0.96      0.94      1155
weighted avg       0.98      0.98      0.98      1155


=== Classification Report — Test ===
              precision    recall  f1-score   support

   No nódulo       0.99      0.99      0.99      1830
      Nódulo       0.91      0.86      0.88       183

    accuracy                           0.98      2013
   macro avg       0.95      0.92      0.94      2013
weighted avg       0.98      0.98      0.98      2013



## 5. Visualizaciones

### 5.1 Curvas ROC (Val y Test)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 7))

for labels, probs, name, color in [
    (val_labels, val_probs, 'Validación', '#2196F3'),
    (test_labels, test_probs, 'Test', '#FF5722'),
]:
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC={auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curva ROC — Expert 3 DenseNet3D')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardada: {OUTPUT_DIR / "roc_curves.png"}')

### 5.2 Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, labels, preds, title in [
    (axes[0], val_labels, val_preds, 'Validación'),
    (axes[1], test_labels, test_preds, 'Test'),
]:
    cm = confusion_matrix(labels, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['No nódulo', 'Nódulo'],
        yticklabels=['No nódulo', 'Nódulo'],
        annot_kws={'size': 16},
    )
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
    ax.set_title(f'Matriz de Confusión — {title}')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardada: {OUTPUT_DIR / "confusion_matrices.png"}')

### 5.3 Evolución de métricas durante entrenamiento

In [ ]:
with open(LOG_PATH) as f:
    train_log = json.load(f)

epochs = [e['epoch'] for e in train_log]
train_losses = [e['train_loss'] for e in train_log]
val_losses = [e['val_loss'] for e in train_log]
val_f1s = [e['val_f1_macro'] for e in train_log]
val_aucs = [e['val_auc'] for e in train_log]
lrs = [e['lr'] for e in train_log]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(epochs, train_losses, 'o-', label='Train Loss', color='#2196F3')
axes[0, 0].plot(epochs, val_losses, 's-', label='Val Loss', color='#FF5722')
axes[0, 0].set_xlabel('Época')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Train vs Val Loss')
axes[0, 0].legend()

# F1-Macro
axes[0, 1].plot(epochs, val_f1s, 's-', color='#4CAF50', lw=2)
axes[0, 1].set_xlabel('Época')
axes[0, 1].set_ylabel('F1-Macro')
axes[0, 1].set_title('Val F1-Macro')
axes[0, 1].axhline(y=max(val_f1s), color='gray', ls='--', alpha=0.5)
axes[0, 1].annotate(f'Best: {max(val_f1s):.4f}', xy=(epochs[np.argmax(val_f1s)], max(val_f1s)),
                     fontsize=10, ha='center', va='bottom')

# AUC
axes[1, 0].plot(epochs, val_aucs, 's-', color='#9C27B0', lw=2)
axes[1, 0].set_xlabel('Época')
axes[1, 0].set_ylabel('AUC-ROC')
axes[1, 0].set_title('Val AUC-ROC')
axes[1, 0].axhline(y=max(val_aucs), color='gray', ls='--', alpha=0.5)
axes[1, 0].annotate(f'Best: {max(val_aucs):.4f}', xy=(epochs[np.argmax(val_aucs)], max(val_aucs)),
                     fontsize=10, ha='center', va='bottom')

# Learning rate
axes[1, 1].plot(epochs, lrs, 'o-', color='#FF9800', lw=2)
axes[1, 1].set_xlabel('Época')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].ticklabel_format(style='scientific', axis='y', scilimits=(0, 0))

plt.suptitle('Evolución del Entrenamiento — Expert 3 DenseNet3D (23 épocas)', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardada: {OUTPUT_DIR / "training_evolution.png"}')

### 5.4 Distribución de probabilidades predichas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, probs, labels, title in [
    (axes[0], val_probs, val_labels, 'Validación'),
    (axes[1], test_probs, test_labels, 'Test'),
]:
    ax.hist(probs[labels == 0], bins=50, alpha=0.6, label='No nódulo', color='#2196F3', density=True)
    ax.hist(probs[labels == 1], bins=50, alpha=0.6, label='Nódulo', color='#FF5722', density=True)
    ax.axvline(x=0.5, color='black', ls='--', lw=1.5, label='Umbral 0.5')
    ax.set_xlabel('P(nódulo)')
    ax.set_ylabel('Densidad')
    ax.set_title(f'Distribución de probabilidades — {title}')
    ax.legend()

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'prob_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardada: {OUTPUT_DIR / "prob_distributions.png"}')

## 6. Análisis de data drift (Val vs Test)

In [22]:
print('=== Análisis de Data Drift (Val vs Test) ===\n')

# Comparar distribuciones de probabilidades
from scipy import stats

ks_stat, ks_pval = stats.ks_2samp(val_probs, test_probs)
print(f'Kolmogorov-Smirnov test (distribución de probs):')
print(f'  Estadístico: {ks_stat:.4f}')
print(f'  p-valor:     {ks_pval:.4f}')
print(f'  Drift detectado: {"SÍ" if ks_pval < 0.05 else "NO"} (α=0.05)\n')

# Comparar prevalencia
val_prev = val_labels.mean()
test_prev = test_labels.mean()
print(f'Prevalencia de positivos:')
print(f'  Val:  {val_prev:.4f} ({val_labels.sum()}/{len(val_labels)})')
print(f'  Test: {test_prev:.4f} ({test_labels.sum()}/{len(test_labels)})')
print(f'  Δ:    {test_prev - val_prev:+.4f}\n')

# Comparar métricas clave
print('Comparación de métricas clave:')
for metric in ['F1-Macro', 'F1-Binary', 'AUC-ROC', 'Precision', 'Recall']:
    v, t = val_metrics[metric], test_metrics[metric]
    delta = t - v
    flag = ' ⚠️' if abs(delta) > 0.05 else ' ✓'
    print(f'  {metric:12s}: Val={v:.4f}  Test={t:.4f}  Δ={delta:+.4f}{flag}')

print('\n(⚠️ = diferencia > 5 puntos, posible drift o sobreajuste)')

=== Análisis de Data Drift (Val vs Test) ===

Kolmogorov-Smirnov test (distribución de probs):
  Estadístico: 0.0230
  p-valor:     0.8205
  Drift detectado: NO (α=0.05)

Prevalencia de positivos:
  Val:  0.0909 (105/1155)
  Test: 0.0909 (183/2013)
  Δ:    +0.0000

Comparación de métricas clave:
  F1-Macro    : Val=0.9438  Test=0.9353  Δ=-0.0085 ✓
  F1-Binary   : Val=0.8981  Test=0.8820  Δ=-0.0161 ✓
  AUC-ROC     : Val=0.9911  Test=0.9821  Δ=-0.0090 ✓
  Precision   : Val=0.8739  Test=0.9075  Δ=+0.0336 ✓
  Recall      : Val=0.9238  Test=0.8579  Δ=-0.0659 ⚠️

(⚠️ = diferencia > 5 puntos, posible drift o sobreajuste)


## 7. Guardar resultados

In [23]:
# Guardar métricas en JSON
results = {
    'model': 'Expert3DenseNet3D',
    'checkpoint_epoch': int(ckpt['epoch']),
    'device': str(DEVICE),
    'validation': {k: float(v) if isinstance(v, (float, np.floating)) else int(v) for k, v in val_metrics.items()},
    'test': {k: float(v) if isinstance(v, (float, np.floating)) else int(v) for k, v in test_metrics.items()},
    'confusion_matrix_val': confusion_matrix(val_labels, val_preds).tolist(),
    'confusion_matrix_test': confusion_matrix(test_labels, test_preds).tolist(),
}

results_path = OUTPUT_DIR / 'evaluation_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Resultados guardados en: {results_path}')

# Guardar tabla como CSV
metrics_df.to_csv(OUTPUT_DIR / 'metrics_summary.csv', index=False)
print(f'Tabla guardada en: {OUTPUT_DIR / "metrics_summary.csv"}')

print('\n=== Evaluación completada ===')
print(f'Figuras y resultados en: {OUTPUT_DIR}/')

Resultados guardados en: /home/nicolas/Escritorio/Academico/analitica_datos/carlos_andres_ferro/proyecto_2/notebooks/evaluacion_expert3/evaluation_results.json
Tabla guardada en: /home/nicolas/Escritorio/Academico/analitica_datos/carlos_andres_ferro/proyecto_2/notebooks/evaluacion_expert3/metrics_summary.csv

=== Evaluación completada ===
Figuras y resultados en: /home/nicolas/Escritorio/Academico/analitica_datos/carlos_andres_ferro/proyecto_2/notebooks/evaluacion_expert3/
